In [18]:
import pandas as pd
import numpy as np

In [19]:
players = pd.read_csv(
    "../data/fifa_players.csv"
)

In [20]:
print(players.columns)

print(players.info())

players.head()

Index(['name', 'full_name', 'birth_date', 'age', 'height_cm', 'weight_kgs',
       'positions', 'nationality', 'overall_rating', 'potential', 'value_euro',
       'wage_euro', 'preferred_foot', 'international_reputation(1-5)',
       'weak_foot(1-5)', 'skill_moves(1-5)', 'body_type',
       'release_clause_euro', 'national_team', 'national_rating',
       'national_team_position', 'national_jersey_number', 'crossing',
       'finishing', 'heading_accuracy', 'short_passing', 'volleys',
       'dribbling', 'curve', 'freekick_accuracy', 'long_passing',
       'ball_control', 'acceleration', 'sprint_speed', 'agility', 'reactions',
       'balance', 'shot_power', 'jumping', 'stamina', 'strength', 'long_shots',
       'aggression', 'interceptions', 'positioning', 'vision', 'penalties',
       'composure', 'marking', 'standing_tackle', 'sliding_tackle'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 17954 entries, 0 to 17953
Data columns (total 51 columns):
 #   Column            

,name,full_name,birth_date,age,height_cm,weight_kgs,positions,nationality,overall_rating,potential,...,long_shots,aggression,interceptions,positioning,vision,penalties,composure,marking,standing_tackle,sliding_tackle
0,L. Messi,Lionel Andrés Messi Cuccittini,6/24/1987,31,170.18,72.1,"CF,RW,ST",Argentina,94,94,...,94,48,22,94,94,75,96,33,28,26
1,C. Eriksen,Christian Dannemann Eriksen,2/14/1992,27,154.94,76.2,"CAM,RM,CM",Denmark,88,89,...,89,46,56,84,91,67,88,59,57,22
2,P. Pogba,Paul Pogba,3/15/1993,25,190.50,83.9,"CM,CAM",France,88,91,...,82,78,64,82,88,82,87,63,67,67
3,L. Insigne,Lorenzo Insigne,6/4/1991,27,162.56,59.0,"LW,ST",Italy,88,88,...,84,34,26,83,87,61,83,51,24,22
4,K. Koulibaly,Kalidou Koulibaly,6/20/1991,27,187.96,88.9,CB,Senegal,88,91,...,15,87,88,24,49,33,80,91,88,87


In [21]:
players = players.dropna(
    subset=[
        "nationality",
        "overall_rating"
    ]
)

In [22]:
players["attack_score"] = (
    players["finishing"] * 0.30 +
    players["shot_power"] * 0.20 +
    players["positioning"] * 0.20 +
    players["volleys"] * 0.15 +
    players["dribbling"] * 0.15
)

In [23]:
players["midfield_score"] = (
    players["vision"] * 0.25 +
    players["short_passing"] * 0.30 +
    players["long_passing"] * 0.20 +
    players["ball_control"] * 0.15 +
    players["composure"] * 0.10
)

In [24]:
players["defense_score"] = (
    players["marking"] * 0.25 +
    players["standing_tackle"] * 0.35 +
    players["sliding_tackle"] * 0.20 +
    players["interceptions"] * 0.20
)

In [25]:
players["physical_score"] = (
    players["strength"] * 0.25 +
    players["stamina"] * 0.25 +
    players["jumping"] * 0.20 +
    players["balance"] * 0.15 +
    players["agility"] * 0.15
)

In [26]:
players["player_score"] = (
    players["overall_rating"] * 0.35 +
    players["potential"] * 0.10 +
    players["attack_score"] * 0.15 +
    players["midfield_score"] * 0.15 +
    players["defense_score"] * 0.15 +
    players["physical_score"] * 0.05 +
    (
        players["international_reputation(1-5)"] * 20
    ) * 0.05
)

In [27]:
players = players.sort_values(
    "player_score",
    ascending=False
)

In [28]:
TOP_PLAYERS = 26

top_players = players.groupby(
    "nationality"
).head(TOP_PLAYERS)

In [29]:
team_stats = top_players.groupby(
    "nationality"
).agg({
    "overall_rating": "mean",
    "potential": "mean",
    "attack_score": "mean",
    "midfield_score": "mean",
    "defense_score": "mean",
    "physical_score": "mean",
    "international_reputation(1-5)": "mean",
    "player_score": "mean",
    "value_euro": "sum"
}).reset_index()

In [30]:
team_stats.columns = [
    "team",
    "avg_overall",
    "avg_potential",
    "attack_score",
    "midfield_score",
    "defense_score",
    "physical_score",
    "international_reputation",
    "avg_player_score",
    "squad_value"
]

In [31]:
team_stats["team_power_score"] = (
    team_stats["avg_player_score"] * 0.40 +
    team_stats["avg_overall"] * 0.20 +
    team_stats["attack_score"] * 0.10 +
    team_stats["midfield_score"] * 0.10 +
    team_stats["defense_score"] * 0.10 +
    team_stats["international_reputation"] * 0.05 +
    (
        team_stats["squad_value"] / 1000000000
    ) * 0.05
)

In [32]:
team_stats = team_stats.sort_values(
    "team_power_score",
    ascending=False
)

In [33]:
team_stats.head(20)

,team,avg_overall,avg_potential,attack_score,midfield_score,defense_score,physical_score,international_reputation,avg_player_score,squad_value,team_power_score
133,Spain,84.423077,85.730769,70.605769,82.809615,75.228846,72.390385,2.884615,78.921923,844000000.0,71.504238
18,Brazil,84.653846,85.923077,72.032692,80.384615,73.048077,74.688462,3.000000,78.775385,871300000.0,71.181027
57,Germany,82.692308,84.500000,70.998077,80.280769,70.951923,72.240385,2.884615,77.223558,744500000.0,69.832417
53,France,83.115385,86.500000,71.609615,78.319231,70.576923,76.351923,2.538462,77.172308,890000000.0,69.714000
119,Portugal,81.038462,83.423077,71.565385,78.113462,69.734615,73.400000,2.615385,75.903173,559300000.0,68.669042
76,Italy,81.115385,83.000000,69.653846,77.450000,71.732692,72.263462,2.384615,75.513654,537000000.0,68.458273
44,England,81.115385,83.076923,70.305769,77.405769,69.292308,73.742308,2.346154,75.281923,594500000.0,68.183263
6,Argentina,82.461538,83.384615,73.530769,78.594231,59.730769,73.788462,2.692308,75.360096,738000000.0,67.993438
13,Belgium,81.500000,83.307692,71.607692,77.836538,64.890385,72.784615,2.461538,75.106731,683000000.0,67.933381
104,Netherlands,79.769231,82.038462,68.471154,76.219231,67.430769,74.367308,2.153846,73.813462,476500000.0,66.822863


In [34]:
team_stats.to_csv(
    "../data/team_stats.csv",
    index=False
)

print("team_stats.csv salvo!")

team_stats.csv salvo!
